<a href="https://colab.research.google.com/github/GulsumSayin/satellite-based-water-change-detection/blob/main/water_monitoring_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Gerekli Kütüphanelerin İmport Edilmesi**

In [ ]:
!pip install gradio

**Drive'a Bağlanma**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from PIL import Image
import io
import os
from datetime import datetime
import tempfile

# ReportLab
!pip install reportlab
!apt-get install fonts-dejavu-core -y

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib import colors as rl_colors
from reportlab.lib.utils import ImageReader

# Fontları kaydet
pdfmetrics.registerFont(TTFont('DejaVu', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
pdfmetrics.registerFont(TTFont('DejaVu-Bold', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))

# ==========================
# MODEL SINIFI
# ==========================
class WaterSegmentationModel:
    def __init__(self, model_path):
        self.model = tf.keras.models.load_model(model_path, compile=False)

    def preprocess(self, img: Image.Image) -> np.ndarray:
        if img.mode != "RGB":
            img = img.convert("RGB")
        img = img.resize((256, 256))
        arr = np.array(img, dtype=np.float32) / 255.0
        return np.expand_dims(arr, axis=0)

    def segment(self, img: Image.Image) -> np.ndarray:
        arr = self.preprocess(img)
        pred = self.model.predict(arr)[0, :, :, 0]
        mask = (pred > 0.5).astype(np.uint8)
        return mask

    def save_mask_image(self, mask: np.ndarray, cmap: str = "Blues", title: str = "") -> io.BytesIO:
        buf = io.BytesIO()
        plt.figure(figsize=(6,6))
        plt.imshow(mask, cmap=cmap)
        if title:
            plt.title(title, pad=20)
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(buf, format="png", dpi=300)
        plt.close()
        buf.seek(0)
        return buf

# ==========================
# PDF RAPOR SINIFI
# ==========================
class PDFReport:
    def __init__(self, output_path="/content/rapor.pdf"):
        self.output_path = output_path

    def generate(self, alan1, alan2, fark, fark_yuzde, risk, cozum, buffers, konum, yil1, yil2):
        c = canvas.Canvas(self.output_path, pagesize=A4)
        width, height = A4

        # Başlık ve bilgiler
        c.setFont("DejaVu-Bold", 18)
        c.drawCentredString(width/2.0, height-3*cm, "Su Değişim Raporu")
        c.setFont("DejaVu", 10)
        today = datetime.today().strftime("%d/%m/%Y")
        c.drawCentredString(width/2.0, height-4*cm, f"Tarih: {today}")
        c.setFont("DejaVu", 12)
        c.drawCentredString(width/2.0, height-4.8*cm, f"Bölge: {konum} ({yil1} vs {yil2})")

        y_pos = height - 6*cm
        line_gap = 0.6*cm
        c.setFont("DejaVu", 11)

        # Alan bilgileri
        c.drawString(2*cm, y_pos, f"1. Görsel Su Alanı: {alan1} piksel")
        y_pos -= line_gap
        c.drawString(2*cm, y_pos, f"2. Görsel Su Alanı: {alan2} piksel")
        y_pos -= line_gap
        c.drawString(2*cm, y_pos, f"Fark: {fark} piksel (%{fark_yuzde:.2f})")

        y_pos -= 1.2*cm
        c.setFillColor(rl_colors.HexColor("#064273"))
        c.drawString(2*cm, y_pos, "Risk Seviyesi:")
        c.setFillColor(rl_colors.black)
        c.drawString(6*cm, y_pos, risk)

        y_pos -= line_gap
        c.setFillColor(rl_colors.HexColor("#064273"))
        c.drawString(2*cm, y_pos, "Çözüm Önerisi:")
        c.setFillColor(rl_colors.black)
        çözüm_text = c.beginText(6*cm, y_pos)
        çözüm_text.setFont("DejaVu", 11)
        for line in cozum.splitlines():
            çözüm_text.textLine(line)
        c.drawText(çözüm_text)

        # Görselleri ekle
        y_pos -= 3.5*cm
        img_width = width - 4*cm
        img_height = 5*cm

        for idx, buf in enumerate(buffers):
            img_path = f"/content/mask_{idx}.png"
            with open(img_path, "wb") as f:
                f.write(buf.getvalue())
            c.drawImage(img_path, 2*cm, y_pos-img_height, img_width, img_height, preserveAspectRatio=True)
            y_pos -= (img_height + 0.5*cm)

        c.showPage()
        c.save()
        return self.output_path


# ==========================
# SU DEĞİŞİM ANALİZ SINIFI
# ==========================
class WaterChangeAnalyzer:
    def __init__(self, model_path):
        self.model_segmenter = WaterSegmentationModel(model_path)
        self.pdf_reporter = PDFReport()

    def load_preview(self, file):
        img = Image.open(file)
        img.thumbnail((256, 256))
        return img

    def analyze(self, file1, file2):
        img1 = Image.open(file1)
        img2 = Image.open(file2)

        # Görüntü isimlerinden info al
        img1_info = os.path.splitext(os.path.basename(file1.name))[0].split('_')
        img2_info = os.path.splitext(os.path.basename(file2.name))[0].split('_')
        konum = img1_info[0].capitalize()
        yil1 = img1_info[1] if len(img1_info) > 1 else "?"
        yil2 = img2_info[1] if len(img2_info) > 1 else "?"

        # Maskeleri üret
        mask1 = self.model_segmenter.segment(img1)
        mask2 = self.model_segmenter.segment(img2)
        fark_mask = mask2.astype(np.float32) - mask1.astype(np.float32)

        # Alan ve fark hesapla
        alan1 = int(np.sum(mask1))
        alan2 = int(np.sum(mask2))
        fark = alan2 - alan1
        fark_yuzde = ((fark) / alan1) * 100 if alan1 != 0 else 0

        # ==========================
        # Yönlü analiz
        # ==========================
        if fark_yuzde == 0:
            risk = "Sabit Durum"
            cozum = (
                "İlgili dönemler arasında su varlığında anlamlı bir değişim gözlemlenmemiştir. "
                "Çevresel istikrar korunmakla birlikte, iklimsel etkiler gecikmeli\n"
                "yansıyabileceğinden izleme periyodik olarak sürdürülmelidir.\n\n"
                "Öneri: NDWI temelli zaman serisi analizleri yapılmalı, Sentinel-2 verisi ile trendlere bakılmalıdır."
            )

        elif fark_yuzde < 0:  # Su azalmış
            fark_yuzde_abs = abs(fark_yuzde)
            if fark_yuzde_abs < 3:
                risk = "Çok Düşük Azalma (Model Belirsizliği)"
                cozum = (
                    "Su alanındaki azalma %3'ün altındadır. Bu değişim model belirsizliği veya mevsimsel etkilerle açıklanabilir.\n\n"
                    "Öneri: Uzun dönem uydu verileriyle karşılaştırmalı analiz yapılmalı, model hassasiyeti gözden geçirilmelidir."
                )
            elif fark_yuzde_abs < 10:
                risk = "Düşük Düzeyde Azalma"
                cozum = (
                    "Su varlığında %3–10 arasında azalma gözlemlenmiştir. Kuraklık, tarımsal\n"
                    "sulama baskısı veya iklimsel faktörlerden kaynaklanabilir.\n\n"
                    "Öneri: Meteorolojik verilerle korelasyon analizi yapılmalı; su politikaları\n"
                    "yerel bazda değerlendirilmelidir."
                )
            elif fark_yuzde_abs < 20:
                risk = "Orta Düzeyde Su Kaybı"
                cozum = (
                    "Su kaynaklarında belirgin azalma tespit edilmiştir.\n"
                    "Bu değişim yapısal ve sürdürülebilirlik riski taşır.\n\n"
                    "Öneri: CORINE/ESA arazi örtüsü ile karşılaştırmalı analiz yapılmalı;\n"
                    "SWAT gibi hidrolojik modellerle senaryo analizi yürütülmelidir."
                )
            else:
                risk = "Yüksek Düzeyde Su Kaybı"
                cozum = (
                    "Su alanında kritik seviyede azalma söz konusudur. Bu durum çevresel çöküş,\n"
                    "kuraklık ve toplumsal etki yaratabilir.\n\n"
                    "Öneri: Saha çalışmaları, radar destekli doğrulama, MODIS NDVI analizi\n"
                    "ve kurumlar arası veri paylaşımıyla entegre analiz yapılmalıdır."
                )

        else:  # fark_yuzde > 0, Su artmış
            if fark_yuzde < 5:
                risk = "Düşük Düzeyde Artış"
                cozum = (
                    "Su alanında sınırlı artış gözlemlenmiştir.\n"
                    "Bu durum normal mevsimsel birikim olabilir.\n\n"
                    "Öneri: Artış trendi izlenmeli, baraj/gölet\n"
                    "hacim verileriyle ilişkilendirilmelidir."
                )
            elif fark_yuzde < 15:
                risk = "Orta Düzeyde Artış"
                cozum = (
                    "Su alanında belirgin artış tespit edilmiştir. Bu durum, olası taşkın,\n"
                    "su birikimi veya altyapı etkisini gösterebilir.\n\n"
                    "Öneri: Sentinel-1 radar verisiyle yüzey suyu yayılımı kontrol edilmeli,\n"
                    "topografik analizle taşkın risk haritası çıkarılmalıdır."
                )
            else:
                risk = "Yüksek Düzeyde Artış (Taşkın Riski)"
                cozum = (
                    "Su alanında %15'in üzerinde artış tespit edilmiştir.\n"
                    "Bu durum, taşkın, sel veya baraj taşması gibi riskler barındırır.\n\n"
                    "Öneri: Sentinel-1 verisiyle doğrulama yapılmalı, taşkın modellemesi\n"
                    "için HEC-RAS, Lisflood gibi sistemler kullanılmalıdır. "
                    "Ayrıca afet yönetimi ile koordineli izleme yapılması önerilir."
                )

        # ==========================
        # Yorum metni
        # ==========================
        yorum = (
            f"Bölge: {konum}\n"
            f"Yıllar: {yil1} vs {yil2}\n\n"
            f"1. Görsel Su Alanı: {alan1} piksel\n"
            f"2. Görsel Su Alanı: {alan2} piksel\n"
            f"Fark: {fark} piksel (%{fark_yuzde:.2f})\n\n"
            f"🛡 Risk Seviyesi: {risk}\n\n"
            f"💡 Çözüm Önerisi: {cozum}"
        )

        # Görselleri kaydet
        buf1 = self.model_segmenter.save_mask_image(mask1, cmap="Blues", title="1. Görsel Su Maskesi")
        buf2 = self.model_segmenter.save_mask_image(mask2, cmap="Blues", title="2. Görsel Su Maskesi")
        buf3 = self.model_segmenter.save_mask_image(fark_mask, cmap="seismic", title="Değişim Alanı Haritası")

        # PDF üret
        pdf_path = self.pdf_reporter.generate(
            alan1, alan2, fark, fark_yuzde, risk, cozum,
            [buf1, buf2, buf3], konum, yil1, yil2
        )

        # Yan yana görselleştirme
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        axs[0].imshow(mask1, cmap="Blues")
        axs[0].set_title("1. Görsel Su Maskesi", pad=20)
        axs[0].axis("off")
        axs[1].imshow(mask2, cmap="Blues")
        axs[1].set_title("2. Görsel Su Maskesi", pad=20)
        axs[1].axis("off")
        axs[2].imshow(fark_mask, cmap="seismic", norm=colors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
        axs[2].set_title("Değişim Alanı Haritası", pad=20)
        axs[2].axis("off")
        plt.subplots_adjust(wspace=0.3)
        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=300)
        plt.close(fig)
        buf.seek(0)
        yan_yana_gorsel = Image.open(buf)

        return yorum, yan_yana_gorsel, pdf_path

# ==========================
# GRADIO ARAYÜZÜ
# ==========================
analyzer = WaterChangeAnalyzer("/content/drive/MyDrive/water_segmentation_model2.keras")

with gr.Blocks(css="""
    body { background-color: #d0ebff; }
    .gradio-container { background-color: #d0ebff !important; }
    #header-container { background-color: #d0ebff; padding: 20px; border-radius: 10px; position: relative; }
    button.svelte-1ipelgc { background-color: #2b7a78 !important; color: white !important; border: none; padding: 10px 20px; font-size: 16px; border-radius: 8px; }
    .gr-image.svelte-1ipelgc { box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); border-radius: 10px; }
""") as interface:

    gr.Markdown("""
    <div id="header-container" style='text-align:center; position:relative'>
        <h1 style='font-size:38px; color:#064273;'>🌊 Su Kaynağı Takip ve Risk Analiz Sistemi</h1>
        <p style='font-size:16px; color:#333;'>İki farklı tarihli uydu görüntüsünden su varlığı değişimini analiz edin ve detaylı PDF raporu oluşturun.</p>
    </div>
    """)

    with gr.Row():
        file1 = gr.File(label="1. Uydu Görüntüsü Seçin")
        thumb1 = gr.Image(label="1. Görüntü", interactive=False)

    with gr.Row():
        file2 = gr.File(label="2. Uydu Görüntüsü Seçin")
        thumb2 = gr.Image(label="2. Görüntü", interactive=False)

    btn = gr.Button("Segmentasyonu Başlat")

    output_text = gr.Textbox(label="Sonuç, Risk ve Çözüm Önerisi")
    output_img = gr.Image(label="Maske Karşılaştırması")
    output_pdf = gr.File(label="PDF Raporunu İndir")

    file1.change(fn=analyzer.load_preview, inputs=file1, outputs=thumb1)
    file2.change(fn=analyzer.load_preview, inputs=file2, outputs=thumb2)
    btn.click(fn=analyzer.analyze, inputs=[file1, file2], outputs=[output_text, output_img, output_pdf])

interface.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-dejavu-core
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 1,041 kB of archives.
After this operation, 3,025 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-core all 2.37-2build1 [1,041 kB]
Fetched 1,041 kB in 2s (637 kB/s)
Selecting previously unselected package fonts-dejavu-core.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../fonts-dejavu-core_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-core (2.37-2build1) ...
Setting up fonts-dejavu-core (2.37-2build1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


/tmp/ipython-input-3569547196.py:281: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css="""


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://700a9c7ed860aa1c52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
